# 06 — A demanding case: California housing

*Chapter 08 · Gradient Boosting · Notebook 6 of 6*

This is the chapter's **capstone**: a full, honest regression workflow on a real dataset, using everything
we built. We predict California median house values end to end — look at the data, set baselines, tune a
gradient booster **with early stopping**, report **R² and the error in real dollars** on a sealed test, find
**where the model fails**, foil it against a random forest and a linear model, and meet the fast modern
default that bridges to the next two chapters.

**Prerequisites.** All of Chapter 08 (the residual engine; the gradient view; classification; ν / depth /
n_estimators; the estimator and early stopping); Chapter 06 Notebook 5 (permutation importance);
Chapter 00 (train/test, regression).

**What you'll do.**
- Inspect the data honestly (including a built-in artifact).
- Build baselines, then a tuned GB with early stopping.
- Report held-out **R² and MAE in dollars**, and analyze **where** it errs.
- Foil it against a random forest and a linear model.
- Meet **`HistGradientBoostingRegressor`** — the fast modern default, and the bridge to Chapters 09–10.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

from ml_course import viz
from ml_course.colors import CMAP_PROBA, COLORS

viz.use_course_style()
SEED = 0

# Real data: California districts (1990 census). Target = median house value, in units of $100,000.
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
n_capped = int((y >= 4.999).sum())
print(f"shape {X.shape};  features: {list(X.columns)}")
print(f"target (median value, $100k): min {y.min():.2f}  max {y.max():.2f}  mean {y.mean():.3f}")
print(f"capped at $500k (y = 5.00): {n_capped} rows = {100 * n_capped / len(y):.1f}%")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
print(f"train {X_train.shape[0]}  test {X_test.shape[0]}")

## The problem, and an honest first look

We predict a district's **median house value** (in units of $100,000) from eight numeric features —
median income, house age, average rooms, location, and so on. We always look at the data before modelling.

Two things to keep in mind. First, an **artifact**: the target is **capped at $500k** — every district
whose true median was at or above $500k was recorded as exactly 5.0 (the printout above: about 4.8% of
rows). Those labels are **censored**: the true values above the cap were erased and flattened to 5.0, so no
model can *learn* what lies up there, and the errors at the top will suffer for it. Second, two of the
features are pure **geography** — `Latitude` and `Longitude` — which we will watch closely.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(y, bins=50, color=COLORS["model"], edgecolor=COLORS["text"], linewidth=0.4)
ax.axvline(5.0, color=COLORS["error"], linestyle="--", linewidth=1.6, label="$500k cap")
ax.set_xlabel("median house value ($100k)")
ax.set_ylabel("count")
ax.set_title("the target — a spike at the $500k ceiling")
ax.legend()
plt.show()

**Read the figure.** Most districts fall between $100k and $300k, with a long right tail — and a hard
**spike at 5.0**, the $500k ceiling holding ~4.8% of the data. That cap is not real signal: it is a
recording limit. The true values above it were erased, so no model can *learn* them — and we will see the
cap bend the errors on expensive homes.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6.5))
sc = ax.scatter(X["Longitude"], X["Latitude"], c=y, cmap=CMAP_PROBA, s=7, alpha=0.5)
fig.colorbar(sc, ax=ax, label="median house value ($100k)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("California: house value tracks location")
plt.show()

**Read the figure.** Plotting each district at its longitude and latitude redraws the map of
California — and the colour (value) is anything but uniform. The **coast** is dear — the Bay Area, Los
Angeles, and San Diego stand out in deep blue — while the inland **Central Valley** is cheap. Location
clearly carries a lot of the signal. Hold that thought: we will ask the model whether it agrees.

## Baselines first

Good practice never reaches for the complex model first. We fit two simple references — a **linear model**
and a **shallow tree** — on the training set and score them on the **sealed test set**, so we know the bar
that "good" has to clear.

In [ ]:
lin = LinearRegression().fit(X_train, y_train)
tree = DecisionTreeRegressor(max_depth=3, random_state=SEED).fit(X_train, y_train)
for name, model in [("LinearRegression", lin), ("DecisionTree(depth=3)", tree)]:
    pred = model.predict(X_test)
    print(f"{name:22s}: test R² = {r2_score(y_test, pred):.3f}   "
          f"MAE = ${mean_absolute_error(y_test, pred) * 100:.1f}k")

**Read the baselines.** A linear model reaches R² **0.59** — a typical error of about **$53k** — and
a depth-3 tree only **0.50**. Neither is bad, but both leave a lot on the table: the relationship is
non-linear with interactions (value depends on income *and* location together), exactly the territory
gradient boosting owns. Let us beat them.

## A tuned gradient booster

We apply the whole chapter: a **small learning rate** and **shallow trees** (depth 3, the interaction order
from Notebook 4), and **early stopping** (Notebook 5) to choose the number of trees from a held-out slice —
no guessing. We ask for up to 2000 trees and let the validation score decide where to stop.

In [ ]:
gb_default = GradientBoostingRegressor(random_state=SEED).fit(X_train, y_train)
gb = GradientBoostingRegressor(
    n_estimators=2000, learning_rate=0.1, max_depth=3,
    validation_fraction=0.1, n_iter_no_change=15, tol=1e-4, random_state=SEED,
).fit(X_train, y_train)
n_stop = gb.n_estimators_
pred = gb.predict(X_test)          # the headline model, reused below
yte = y_test.to_numpy()

# A full (no-early-stop) model, to trace the staged test-R² curve past the stop.
gb_full = GradientBoostingRegressor(
    n_estimators=800, learning_rate=0.1, max_depth=3, random_state=SEED
).fit(X_train, y_train)
test_r2 = np.array([r2_score(y_test, p) for p in gb_full.staged_predict(X_test)])

print(f"GB default (100 trees) : test R² {gb_default.score(X_test, y_test):.3f}  "
      f"MAE ${mean_absolute_error(y_test, gb_default.predict(X_test)) * 100:.1f}k")
print(f"GB early-stopped       : used {n_stop} trees, test R² {gb.score(X_test, y_test):.3f}  "
      f"MAE ${mean_absolute_error(y_test, pred) * 100:.1f}k")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(1, len(test_r2) + 1), test_r2, color=COLORS["model"], linewidth=2,
        label="test R²")
ax.axvline(n_stop, color=COLORS["highlight"], linestyle="--", linewidth=1.8,
           label=f"early stop: {n_stop} trees")
ax.set_xlabel("number of trees")
ax.set_ylabel("test R²")
ax.set_title("gradient boosting with early stopping on California housing")
ax.legend()
plt.show()

**Read the figure.** A default 100 trees already reaches R² **0.78**. Early stopping watches a
held-out validation slice and halts at about **450 trees** — where the validation gain falls below
tolerance — giving test R² **0.82** and a typical error near **$33k** (down from the baseline's $53k), the
number of trees chosen from the data rather than guessed. The test curve is nearly flat there: it would
creep up only marginally (to ≈ 0.83) over hundreds more trees. (The curve drawn is a full-data model's test
trajectory; the dashed line is where the validation-monitored model stopped — close but not identical, since
early stopping optimizes the small validation slice, not the test set.)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1).fit(X_train, y_train)
hgb = HistGradientBoostingRegressor(random_state=SEED).fit(X_train, y_train)

models = {
    "Linear": lin, "Tree (d3)": tree, "RandomForest": rf,
    "GB (early-stop)": gb, "HistGBR": hgb,
}
names = list(models)
r2s = [r2_score(y_test, m.predict(X_test)) for m in models.values()]
maes = [mean_absolute_error(y_test, m.predict(X_test)) * 100 for m in models.values()]
for n, r, ma in zip(names, r2s, maes, strict=True):
    print(f"{n:16s}: R² {r:.3f}   MAE ${ma:.1f}k")

bar_colors = [
    COLORS["muted"], COLORS["muted"], COLORS["class_c"], COLORS["model"], COLORS["highlight"],
]
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.5))
axL.bar(names, r2s, color=bar_colors, edgecolor=COLORS["text"], linewidth=0.6)
axL.set_ylabel("test R²")
axL.set_ylim(0, 1)
axL.tick_params(axis="x", rotation=30)
axL.set_title("test R² (higher is better)")
axR.bar(names, maes, color=bar_colors, edgecolor=COLORS["text"], linewidth=0.6)
axR.set_ylabel("MAE ($k)")
axR.tick_params(axis="x", rotation=30)
axR.set_title("mean absolute error (lower is better)")
fig.tight_layout()
plt.show()

**Read the figure.** The ladder is clear: linear **0.59** → tree **0.50** → random forest **0.80** →
tuned GB **0.82** → **HistGBR 0.84**. Notice the honesty in the detail — the forest beats the *default* GB,
but the *tuned* GB beats the forest, and the histogram booster tops both. The four strong models are
**close**: there is **no universal best**, the winner tracks the dataset, and here the modern histogram
booster leads. We report both R² and MAE (in dollars), because the dollar error is what a person actually
feels.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(y_test, pred, s=8, alpha=0.3, color=COLORS["model"])
lims = [0, 5.7]
ax.plot(lims, lims, color=COLORS["error"], linestyle="--", linewidth=1.4,
        label="perfect prediction")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("actual value ($100k)")
ax.set_ylabel("predicted value ($100k)")
ax.set_title("predicted vs actual (tuned gradient boosting)")
ax.legend()
plt.show()

**Read the figure.** In the bulk the cloud hugs the diagonal — predictions track reality. Two honest
flaws show at the edges: a **vertical wall at actual = 5.0** — the capped homes, whose true value is pinned
at the ceiling while the model spreads its guesses around it (a few even poke *above* 5.0: a sum of trees is
not bounded by the training maximum — the real limit is that the values above the cap were never shown to
it) — and a **wider spread** for expensive houses. A single R² hides this; the picture does not.

## Where does it err?

A single error number averages over very different homes. Let us break the mean absolute error down by the
true price of the home.

In [ ]:
buckets = [(0.0, 2.0), (2.0, 3.5), (3.5, 4.5), (4.5, 5.01)]
labels, bucket_mae = [], []
for lo, hi in buckets:
    mask = (yte >= lo) & (yte < hi)
    labels.append(f"{lo:.1f}–{min(hi, 5.0):.1f}")
    bucket_mae.append(mean_absolute_error(yte[mask], pred[mask]) * 100)
    print(f"price [{lo}, {hi}) n={int(mask.sum()):4d}: MAE ${bucket_mae[-1]:.1f}k")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.bar(labels, bucket_mae, color=COLORS["model"], edgecolor=COLORS["text"], linewidth=0.6)
ax.set_xlabel("true median value bucket ($100k)")
ax.set_ylabel("MAE ($k)")
ax.set_title("the model errs far more on expensive homes")
plt.show()

**Read the figure.** The error climbs steeply with price: about **$25k** on the cheapest homes to
about **$74k** on the priciest bucket — roughly **three times** worse. The error already doubles in the
$350k–$450k bucket, which sits *below* the cap — so rarity and heterogeneity (expensive homes are fewer and
more varied) are doing real work before censoring even applies; the $500k cap then bites hardest on the top
bucket (where the model is partly right to hedge). The headline "$33k MAE" is an average over very unequal
regions — always look underneath it.

## Which features did it use?

The model exposes its `feature_importances_` (MDI — impurity reduction). As in Chapter 06, we never trust
MDI alone: we cross-check with **permutation importance** on the held-out test set. The map (Figure 2) gave
us a strong prior about what *should* matter.

In [ ]:
mdi = gb.feature_importances_
perm = permutation_importance(gb, X_test, y_test, n_repeats=10, random_state=SEED)
print("feature        MDI    permutation")
for i in np.argsort(perm.importances_mean)[::-1]:
    print(f"  {X.columns[i]:11s} {mdi[i]:.3f}   {perm.importances_mean[i]:.3f}")

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.5))
viz.plot_feature_importances(X.columns, mdi, top=8, ax=axL, label="MDI")
axL.set_title("MDI (impurity) — income looks dominant")
viz.plot_feature_importances(X.columns, perm.importances_mean, std=perm.importances_std,
                             top=8, ax=axR, color=COLORS["highlight"], label="permutation")
axR.set_title("permutation (held-out) — location dominates")
fig.tight_layout()
plt.show()

**Read the figure — a genuine disagreement.** By **MDI**, median income dominates (≈ 0.58) and the
two location features look minor (≈ 0.10 each). **Permutation importance tells the opposite story**:
shuffling `Latitude` or `Longitude` *collapses* the model (a drop of ≈ 3.2–3.4 in R², far past zero), so
**location is the dominant factor**, with income second.

Why the gap? In Notebook 5 we noted MDI is biased *toward* continuous features — yet here it *under*-credits
the continuous latitude and longitude. The reconciliation is **interactions**. Median income carries strong
**first-order** signal, captured in a few high-impurity splits near the top of each tree, which MDI rewards
richly. Location's signal is **interaction-borne** — value depends on latitude *and* longitude *and* income
together (the depth-3 interactions of Notebook 4) — so it is spread across **many deep, low-impurity
splits**, each of which MDI scores as tiny, making the total look small. Permutation does not care how the
signal is distributed: it shuffles a column and watches the held-out score fall, so destroying location
destroys the whole interaction and the model collapses. **On held-out data, permutation is the read to trust
here** (Chapter 06), and it agrees with the **map** (Figure 2). Two cautions: `Latitude` and `Longitude` are
themselves correlated, so read the *pair* together rather than as two independent verdicts; and importance is
**use, not cause** — location is a *proxy* for unmeasured neighbourhood quality, schools, and access.

## The fast modern default, and the bridge ahead

Look back at the comparison: `HistGradientBoostingRegressor` reached the **best** R² (≈ 0.84), and **several
times faster** than the classic booster. It bins features into histograms (so split-finding scans a few
hundred bins, not every sorted value), grows trees under a leaf-node budget, regularizes its objective, and
handles missing values natively. That recipe is
exactly what **XGBoost (Chapter 09)** and **LightGBM (Chapter 10)** refine and scale. The engine you built
by hand in Notebook 1 is the same engine inside those state-of-the-art libraries — they add speed and
regularization, not a new idea.

## Your turn

1. **Read the results.** From the comparison figure, name the best model and its typical dollar error; from
   the error-by-price figure, say which homes the model handles worst, and give one reason.
2. **Tune a little more.** Re-fit the gradient booster with `max_depth=4` and `learning_rate=0.05` (keep
   early stopping). Does the sealed-test R² beat 0.82? Relate `max_depth` to the interaction order of
   Notebook 4.
3. **Explain the divergence.** In a few sentences, explain why MDI ranks `MedInc` first while permutation
   ranks `Latitude`/`Longitude` first. Using the map (Figure 2), argue whether location *should* be
   important — then say why "important to the model" still does not mean "causes the price".

## What you built — and the chapter you finished

- You ran a **full honest regression workflow**: looked at the data (and its $500k cap), set baselines,
  tuned a gradient booster **with early stopping**, reported **R² and MAE in dollars** on a sealed test,
  analyzed **where** it errs, and foiled it against a forest, a linear model, and the fast `HistGradientBoosting`.
- You met a real **MDI-vs-permutation disagreement** and trusted the held-out measurement — importance is
  use, not cause.

And with that you have closed **Chapter 08**: from fitting residuals by hand (Notebook 1), to seeing those
residuals *are* a gradient (Notebook 2), to classification (Notebook 3), to controlling complexity
(Notebook 4), to the real estimator and early stopping (Notebook 5), to a competitive, well-tuned, honestly
evaluated model on real data (here). **Vocabulary:** baseline-first · held-out R² / MAE in units · error
analysis by segment · MDI-vs-permutation divergence · `HistGradientBoosting*`.

## Going further (optional)

- The **$500k cap** is censored data; honest practice reports the limitation (or models the censoring
  explicitly). `loss='huber'` or `loss='quantile'` give robust or interval predictions for the heavy tail.
- **Location** matters through interactions (latitude × longitude × income); boosting captures it via tree
  depth (Notebook 4). Engineered features (distance to coast, to a city) would help a linear model close
  the gap.
- A production workflow would tune on a validation split or with nested cross-validation rather than a
  single fit, and would watch for **spatial leakage** (nearby districts in both train and test).
- The speed and regularization that let `HistGradientBoosting*` win here are the subject of **Chapters 09
  (XGBoost)** and **10 (LightGBM)**.

## References

- Pace, R. K., & Barry, R. (1997). *Sparse spatial autoregressions.* Statistics & Probability Letters,
  33(3), 291–297. DOI [10.1016/S0167-7152(96)00140-X](https://doi.org/10.1016/S0167-7152(96)00140-X).
  (The California housing dataset.)
- Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* Annals of
  Statistics, 29(5), 1189–1232. DOI [10.1214/aos/1013203451](https://doi.org/10.1214/aos/1013203451).
- Friedman, J. H. (2002). *Stochastic gradient boosting.* CSDA, 38(4), 367–378. DOI
  [10.1016/S0167-9473(01)00065-2](https://doi.org/10.1016/S0167-9473(01)00065-2).
- Breiman, L. (2001). *Random forests.* Machine Learning, 45(1), 5–32. DOI
  [10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324). (Permutation importance.)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10.11–10.13. DOI [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).
- *Forward:* Chen & Guestrin (2016). *XGBoost.* DOI
  [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785); Ke et al. (2017). *LightGBM.* (NeurIPS).

*Previous: 05 — The estimator and its parameters.*  ·  *Next: Chapter 09 — XGBoost.*